# Практическая работа №1. Часть 1С
## Глубокая сеть: слой, активация, XOR

**Цель:** написать прямой проход полносвязной нейросети на «голых» тензорах PyTorch, убедиться, что без нелинейности сеть не может решить задачу XOR, а с активацией ReLU решает, и сверить свой код с готовыми слоями `torch.nn`.

> **Правило части.** Внутри своих функций – только операции тензоров: `@`, `+`, `-`, `*`, `/`, `torch.maximum`, `torch.exp`, `torch.log`, `.sum`, `.max`. Готовые `nn.Linear`, `torch.relu`, `torch.softmax`, `F.cross_entropy` внутри функций использовать нельзя. Модуль `torch.nn` разрешен только в задании С5 (сверка) и в готовой ячейке обучения в задании С6.

**Задача:** сквозной пример – функция XOR на четырех точках и веса сети из книги Гудфеллоу (раздел 6.1). В конце посмотрим, какую границу решений рисует сеть на данных «две луны».

**Как работать с ноутбуком.** В ячейках с заданиями замените `...` своим кодом – подсказка написана в комментарии `# <--`. После функций идут строки проверки: напечатанное значение должно совпасть с тем, что стоит после слова «ожидается». Готовые ячейки (без `...`) просто выполните. Сначала прочитайте теорию ниже – все задания опираются на нее.

## Как устроена полносвязная сеть

**1. Нейрон – это знакомая логистическая регрессия.** В части 4 практической работы 9 класса логистическая регрессия считала для объекта с признаками $x_1, \dots, x_n$ граничную функцию и пропускала ее через сигмоиду:

$$z = w_1x_1 + w_2x_2 + \dots + w_nx_n + b, \qquad h = \sigma(z).$$

Нейрон устроен точно так же, только вместо сигмоиды может стоять любая **функция активации** $f$: $h = f(z)$. Вес $w_j$ показывает, насколько сильно и в какую сторону признак $x_j$ влияет на ответ, смещение $b$ сдвигает порог срабатывания. Число $z$ до активации часто называют **предактивацией**, а $h$ – **активацией** нейрона.

**2. Слой – много нейронов, посчитанных одним умножением.** Пусть в слое $m$ нейронов и у каждого свой набор из $n$ весов. Запишем веса нейрона $j$ в $j$-й **столбец** матрицы $W$ размера $n \times m$, а смещения – в вектор $b$ длины $m$. Тогда выходы всех нейронов для одного объекта – это строка

$$z = xW + b, \qquad z_j = x_1W_{1j} + x_2W_{2j} + \dots + x_nW_{nj} + b_j.$$

Если объектов сразу $N$ (такой набор называется **пакетом**, batch), сложим их строками в матрицу $X$ размера $N \times n$. Одно умножение матриц посчитает все нейроны на всех объектах:

$$Z = XW + b, \qquad \underbrace{(N \times n)}_{X} \cdot \underbrace{(n \times m)}_{W} \to \underbrace{(N \times m)}_{Z}.$$

Строка $Z$ – один объект, столбец – один нейрон. Вектор $b$ длины $m$ прибавляется к **каждой** строке – PyTorch делает это сам, это называется **транслированием** (broadcasting). Главное правило форм: у умножаемых матриц внутренние размеры должны совпадать ($n$ и $n$), иначе PyTorch выдаст ошибку `mat1 and mat2 shapes cannot be multiplied`.

*Пример.* Четыре точки XOR – это $X$ размера $4 \times 2$. Слой из двух нейронов имеет $W$ размера $2 \times 2$ и $b$ длины 2, результат – $4 \times 2$.

**3. Зачем нужна активация.** Поставим два слоя друг за другом **без** активации:

$$(xW_1 + b_1)W_2 + b_2 = x\underbrace{(W_1W_2)}_{W'} + \underbrace{(b_1W_2 + b_2)}_{b'}.$$

Получился снова один линейный слой с весами $W'$ и смещением $b'$. Сколько бы линейных слоев мы ни ставили, сеть останется линейной функцией, а граница между классами – прямой (плоскостью). Чтобы слои давали что-то новое, между ними ставят **нелинейную** функцию, которая применяется к каждому числу отдельно:

- $\mathrm{ReLU}(z) = \max(0, z)$ – обнуляет отрицательные числа, положительные пропускает без изменений. Стандартный выбор для скрытых слоев: считается мгновенно и не «сплющивает» большие значения;
- $\sigma(z) = \dfrac{1}{1 + e^{-z}}$ – сигмоида, сжимает любое число в интервал $(0, 1)$;
- $\tanh z = \dfrac{e^{z} - e^{-z}}{e^{z} + e^{-z}}$ – гиперболический тангенс, сжимает в $(-1, 1)$ и симметричен относительно нуля.

Излом ReLU в нуле и есть тот «изгиб», который позволяет сети рисовать непрямые границы.

**4. Задача XOR.** Функция XOR («исключающее ИЛИ») равна 1, если ровно один из двух входов равен 1:

| $x_1$ | $x_2$ | $y$ |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

*Почему линейная модель не справляется.* Пусть $f(x) = a_1x_1 + a_2x_2 + d$. Из $f(0, 0) = 0$ получаем $d = 0$, из $f(0, 1) = 1$ – $a_2 = 1$, из $f(1, 0) = 1$ – $a_1 = 1$. Но тогда $f(1, 1) = 2$, а нужно 0. Противоречие: ни одна прямая не отделит точки $(0, 1)$, $(1, 0)$ от $(0, 0)$, $(1, 1)$.

*Как справляется сеть.* Возьмем скрытый слой из двух нейронов с ReLU и один выходной нейрон без активации:

$$h = \mathrm{ReLU}(xW + c), \qquad \hat y = h\,w + b.$$

С весами из книги $W = \begin{pmatrix}1 & 1\\ 1 & 1\end{pmatrix}$, $c = (0, -1)$, $w = (1, -2)$, $b = 0$ скрытый слой переводит обе точки $(0, 1)$ и $(1, 0)$ в одну и ту же точку $h = (1, 0)$. В новом пространстве $h$ классы уже разделяются прямой – и выходной нейрон, обычная линейная модель, справляется. Главная идея глубокого обучения: **скрытые слои сами строят признаки**, в которых задача становится простой. В 9 классе такие признаки придумывал человек (например, произведение $x_1x_2$ решило бы XOR для логистической регрессии).

**5. Выходной слой и функция потерь.** Последний слой выбирается по задаче:

- **регрессия** (предсказать число): один выход без активации, потеря – среднеквадратичная ошибка $\frac1N\sum(\hat y_i - y_i)^2$;
- **два класса**: один выход $z$, вероятность класса 1 равна $\sigma(z)$, потеря – бинарная кросс-энтропия (тот же log-loss, что в части 4);
- **$K$ классов**: $K$ выходов $z_1, \dots, z_K$ – их называют **логитами** («очками» классов). Функция **softmax** превращает очки в вероятности:

$$p_k = \mathrm{softmax}(z)_k = \frac{e^{z_k}}{e^{z_1} + e^{z_2} + \dots + e^{z_K}}.$$

Экспонента делает все числа положительными, деление на сумму – сумму равной 1, а порядок сохраняется: у кого больше очков, у того больше вероятность. Потеря для объекта с верным классом $y$ – **кросс-энтропия**

$$L = -\ln p_y.$$

Смысл: насколько сеть «удивлена» верным ответом. Если $p_y = 1$, потеря 0; если $p_y \to 0$, потеря растет до бесконечности – уверенная ошибка стоит очень дорого. Для двух классов это в точности log-loss из части 4.

**6. Число параметров.** У слоя $n_{in} \to n_{out}$ матрица весов $n_{in} \times n_{out}$ и $n_{out}$ смещений, всего $n_{in} \cdot n_{out} + n_{out}$ параметров. У сети – сумма по слоям. Например, сеть 784 → 128 → 10 для картинок $28 \times 28$ имеет $784 \cdot 128 + 128 + 128 \cdot 10 + 10 = 101\,770$ параметров.

**7. Почему «глубокая».** Теорема об универсальной аппроксимации утверждает: сеть даже с одним скрытым слоем может сколь угодно точно приблизить любую непрерывную функцию – но нейронов может понадобиться очень много. Сеть с ReLU – кусочно-линейная функция: в каждой области, где набор «включенных» нейронов не меняется, она линейна. Каждый новый слой может удвоить число линейных кусков, поэтому глубокие сети выражают сложные функции экономнее широких (Гудфеллоу, раздел 6.4.1). Граница такой сети состоит из отрезков – это видно в задании С6.

In [ ]:
# Нужна, только если ноутбук запускается отдельно от ПР
import numpy as np
import matplotlib
matplotlib.use('Agg')   # графики сохраняются в .png и показываются через display; для интерактива закомментируйте
import matplotlib.pyplot as plt
import torch
from torch import nn
from sklearn.datasets import make_moons
from IPython.display import Image, display

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
print('torch', torch.__version__)

def show(t, k=3):
    """Печать тензора списком с округлением – так удобно сверять с «ожидается»."""
    t = torch.as_tensor(t).detach()
    if t.dim() == 0:
        return round(t.item(), k)
    return torch.round(t.double() * 10**k).div(10**k).tolist()

# Данные «две луны» генерируются, скачивать ничего не нужно
X_moons_np, y_moons_np = make_moons(n_samples=200, noise=0.2, random_state=RANDOM_STATE)
Xm = torch.tensor(X_moons_np, dtype=torch.float32)
ym = torch.tensor(y_moons_np, dtype=torch.long)
print(Xm.shape, ym.shape, '  ожидается torch.Size([200, 2]) torch.Size([200])')

In [ ]:
# Учебный пример со слайдов: XOR и веса из книги Гудфеллоу (раздел 6.1)
X_xor = torch.tensor([[0., 0.],
                      [0., 1.],
                      [1., 0.],
                      [1., 1.]])
y_xor = torch.tensor([0., 1., 1., 0.])

W = torch.tensor([[1., 1.],
                  [1., 1.]])     # веса скрытого слоя, форма (2, 2) = (n_in, n_out)
c = torch.tensor([0., -1.])      # смещения скрытого слоя
w = torch.tensor([1., -2.])      # веса выходного нейрона
b = torch.tensor(0.)             # смещение выходного нейрона

**Задание С1. Функции активации, softmax и кросс-энтропия.**

Напишите четыре функции. Все они работают с тензорами любой формы и применяются поэлементно (кроме softmax).

1. `relu(z)` – $\max(0, z)$ для каждого элемента. Функция `torch.maximum(a, b)` берет поэлементный максимум двух тензоров **одинаковой формы**, поэтому второй аргумент – тензор нулей той же формы, что `z`: `torch.zeros_like(z)`.
2. `sigmoid(z)` – $\dfrac{1}{1 + e^{-z}}$. Свойства для самопроверки: $\sigma(0) = 0{,}5$ и $\sigma(-z) = 1 - \sigma(z)$. В отличие от `math.exp` из части 4, `torch.exp` не выдает ошибку переполнения: при очень большом аргументе получится `inf`, а $1 / (1 + \infty) = 0$ – это правильный предел.
3. `softmax(z)` – по последней оси (`dim=-1`): для вектора это сам вектор, для матрицы «объекты × классы» – каждая строка отдельно.

   *Проблема переполнения.* Число `float32` не может быть больше примерно $3{,}4 \cdot 10^{38}$, а $e^{89}$ уже больше. Поэтому $e^{1000}$ превратится в `inf`, и деление `inf / inf` даст `nan`. *Решение:* вычесть из всех очков их максимум $m$. Результат не изменится, потому что числитель и знаменатель умножаются на одно и то же число $e^{-m}$:

$$\frac{e^{z_k - m}}{\sum_j e^{z_j - m}} = \frac{e^{-m}\,e^{z_k}}{e^{-m}\sum_j e^{z_j}} = \frac{e^{z_k}}{\sum_j e^{z_j}}.$$

   После вычитания самый большой показатель равен 0, а $e^0 = 1$ – переполнения нет. *Технически:* `z.max(dim=-1, keepdim=True).values` дает максимум в каждой строке; `keepdim=True` сохраняет форму $N \times 1$, чтобы вычитание по правилу транслирования шло построчно. Сумму в знаменателе тоже берите с `keepdim=True`.
4. `cross_entropy(p, k)` – $-\ln p_k$ для вектора вероятностей `p` и номера верного класса `k` (используйте `torch.log`).

*Разбор на числах (пример со слайда).* Очки $z = (2; 1; 0{,}1)$. Экспоненты: $e^2 \approx 7{,}389$, $e^1 \approx 2{,}718$, $e^{0{,}1} \approx 1{,}105$, их сумма $\approx 11{,}213$. Вероятности: $7{,}389 / 11{,}213 \approx 0{,}659$, затем $0{,}242$ и $0{,}099$. Если верен класс 0, потеря $-\ln 0{,}659 \approx 0{,}417$; если класс 2 – $-\ln 0{,}099 \approx 2{,}317$.

In [ ]:
def relu(z):
    return ...  # <-- torch.maximum с нулевым тензором той же формы (torch.zeros_like)

def sigmoid(z):
    return ...  # <-- 1 / (1 + e^(-z))

def softmax(z):
    z = ...     # <-- вычтите максимум по последней оси: z.max(dim=-1, keepdim=True).values
    e = ...     # <-- экспонента
    return ...  # <-- поделите на сумму по последней оси (keepdim=True)

def cross_entropy(p, k):
    return ...  # <-- минус логарифм вероятности верного класса


print(show(relu(torch.tensor([-2., 0., 3.]))), '  ожидается [0.0, 0.0, 3.0]')
print(show(sigmoid(torch.tensor([-2., 0., 2.]))), '  ожидается [0.119, 0.5, 0.881]')
p = softmax(torch.tensor([2., 1., 0.1]))
print(show(p), '  ожидается [0.659, 0.242, 0.099]')
print(show(p.sum()), '  ожидается 1.0')
print(show(softmax(torch.tensor([1000., 1001., 1002.]))), '  ожидается [0.09, 0.245, 0.665] (без nan)')
print(show(softmax(torch.tensor([[2., 1., 0.1], [0., 0., 0.]]))), '  ожидается [[0.659, 0.242, 0.099], [0.333, 0.333, 0.333]]')
print(show(cross_entropy(p, 0)), '  ожидается 0.417')
print(show(cross_entropy(p, 2)), '  ожидается 2.317')

**Вопрос С1.** а) Почему вычитание максимума не меняет результат softmax? б) Сеть дала верному классу вероятность $0{,}659$, а в другой раз – $0{,}099$. Во сколько раз отличаются потери? Почему кросс-энтропия так сильно наказывает уверенную ошибку? в) Чем нейрон с сигмоидой отличается от логистической регрессии из 9 класса?

*Ваш ответ:*

**Задание С2. Линейный слой.**

Функция `linear(x, W, b)` возвращает $xW + b$ (см. п. 2 теории): `x` – матрица $N \times n_{in}$, `W` – $n_{in} \times n_{out}$, `b` – вектор длины $n_{out}$. Умножение матриц в PyTorch записывается оператором `@`, смещение прибавится к каждой строке само.

*Разбор одной строки.* Для точки $x = (0, 1)$ и весов из книги: $xW = 0 \cdot (1, 1) + 1 \cdot (1, 1) = (1, 1)$, затем $+ c = (1, 1) + (0, -1) = (1, 0)$. Так же считаются остальные строки – а `@` делает это для всех сразу.

*Замечание про запись в книге.* У Гудфеллоу объекты – столбцы, и слой записан как $W^\top x + c$. Это та же формула: в PyTorch принято хранить пакет строками, поэтому мы пишем $xW + b$.

In [ ]:
def linear(x, W, b):
    return ...  # <-- матричное умножение @ и сложение со смещением


print(show(linear(X_xor, W, c)), '  ожидается [[0.0, -1.0], [1.0, 0.0], [1.0, 0.0], [2.0, 1.0]]')
print(linear(torch.zeros(64, 784), torch.zeros(784, 128), torch.zeros(128)).shape,
      '  ожидается torch.Size([64, 128])')

**Вопрос С2.** а) Какой формы должны быть `W` и `b` у слоя 784 → 128? б) Что изменится в формулах, если объекты хранить не строками, а столбцами (как в книге Гудфеллоу: $W^\top x + c$)?

*Ваш ответ:*

**Задание С3. Сеть для XOR.**

- `forward_xor(X, W, c, w, b)` возвращает пару `(H, y_hat)`: скрытый слой $H = \mathrm{ReLU}(XW + c)$ и выход $\hat y = Hw + b$. Используйте свои `linear` и `relu`. Так как `w` – вектор длины 2, произведение `H @ w` дает вектор длины 4 – по одному числу на точку.
- `forward_no_act(X, W, c, w, b)` – та же сеть, но без ReLU.

*Что должно получиться (посчитайте на бумаге до запуска).*

| $x$ | $xW + c$ | $h = \mathrm{ReLU}(\cdot)$ | $\hat y = h_1 - 2h_2$ |
|---|---|---|---|
| (0, 0) | (0, −1) | (0, 0) | 0 |
| (0, 1) | (1, 0) | (1, 0) | 1 |
| (1, 0) | (1, 0) | (1, 0) | 1 |
| (1, 1) | (2, 1) | (2, 1) | 0 |

ReLU меняет ровно одно число – $-1$ в первой строке становится 0, – и этого излома хватает, чтобы решить XOR. Без ReLU (п. 3 теории) сеть схлопывается в одну линейную функцию: $Ww = (1 - 2,\; 1 - 2) = (-1, -1)$, $cw + b = 0 \cdot 1 + (-1)(-2) + 0 = 2$, то есть $f(x) = 2 - x_1 - x_2$ и выход $[2, 1, 1, 0]$.

In [ ]:
def forward_xor(X, W, c, w, b):
    H = ...      # <-- relu от linear(X, W, c)
    y_hat = ...  # <-- H @ w + b
    return H, y_hat

def forward_no_act(X, W, c, w, b):
    return ...   # <-- та же сеть без активации


H, y_hat = forward_xor(X_xor, W, c, w, b)
print(show(H), '  ожидается [[0.0, 0.0], [1.0, 0.0], [1.0, 0.0], [2.0, 1.0]]')
print(show(y_hat), '  ожидается [0.0, 1.0, 1.0, 0.0]')
print(show(forward_no_act(X_xor, W, c, w, b)), '  ожидается [2.0, 1.0, 1.0, 0.0]')

In [ ]:
# Готовая ячейка: точки XOR до и после скрытого слоя
fig, axs = plt.subplots(1, 2, figsize=(8, 3.8))
for ax, P, t, lab in [(axs[0], X_xor, 'Входы x', ('$x_1$', '$x_2$')),
                      (axs[1], H, 'Скрытый слой h', ('$h_1$', '$h_2$'))]:
    ax.scatter(P[:, 0], P[:, 1], s=300, c=['#1C1C22' if t_ == 0 else '#B8AB84' for t_ in y_xor],
               edgecolors='#1C1C22')
    ax.set_title(t); ax.set_xlabel(lab[0]); ax.set_ylabel(lab[1])
hh = np.linspace(-0.3, 2.3, 10)
axs[1].plot(hh, (hh - 0.5) / 2, '--', color='#8F8260')   # прямая h1 - 2*h2 = 0.5
fig.tight_layout(); fig.savefig('xor_hidden.png', dpi=120); plt.close(fig)
display(Image('xor_hidden.png'))

**Вопрос С3.** а) Какие две точки скрытый слой «склеил» в одну? Почему после этого хватает одной прямой? б) Сеть без активации выдала $[2, 1, 1, 0]$. Запишите ее как одну линейную функцию $f(x) = a_1x_1 + a_2x_2 + d$. Может ли какая-нибудь линейная функция дать $[0, 1, 1, 0]$?

*Ваш ответ:*

**Задание С4. Сколько параметров у сети.**

Функция `count_params(layer_sizes)` получает список размеров слоев, например `[784, 128, 10]` – вход 784 признака, скрытый слой 128 нейронов, выход 10 классов. Слоев весов на один меньше, чем размеров: каждый соединяет два соседних размера. Пары соседей удобно получить через `zip(layer_sizes[:-1], layer_sizes[1:])` – для `[784, 128, 10]` это `(784, 128)` и `(128, 10)`. Каждый слой дает $n_{in} \cdot n_{out}$ весов и $n_{out}$ смещений.

*Проверка на маленькой сети.* Для XOR `[2, 2, 1]`: первый слой $2 \cdot 2 + 2 = 6$, второй $2 \cdot 1 + 1 = 3$, всего 9 – ровно столько чисел мы задали в `W`, `c`, `w`, `b`.

In [ ]:
def count_params(layer_sizes):
    total = 0
    for n_in, n_out in zip(layer_sizes[:-1], layer_sizes[1:]):
        total += ...  # <-- веса и смещения одного слоя
    return total


print(count_params([784, 128, 10]), '  ожидается 101770')
print(count_params([2, 2, 1]), '  ожидается 9')
print(count_params([2, 16, 16, 1]), '  ожидается 337')

**Вопрос С4.** а) Какой слой сети 784 → 128 → 10 «тяжелее» и во сколько раз? б) Сколько параметров у логистической регрессии из 9 класса на тех же 784 признаках (один класс против остальных)? Во сколько раз сеть больше?

*Ваш ответ:*

**Задание С5. Сверка с `torch.nn`.**

Сеть из нескольких слоев удобно хранить как **список пар** `(W, b)`, по одной на слой: например, сеть XOR – это `[(W, c), (w.reshape(2, 1), b.reshape(1))]`. Выходной вес превращается в матрицу $2 \times 1$, чтобы второй слой имел ту же форму, что и первый.

- `mlp_forward(x, params)` – прямой проход: идем по слоям по порядку, в каждом считаем `linear`, а после каждого слоя, **кроме последнего**, применяем ReLU. На выходе остаются «сырые» числа – логиты. Активацию выходного слоя (сигмоиду или softmax) принято не включать в сеть: ее применяет функция потерь, так вычисления устойчивее.
- `to_sequential(params)` – строит такую же сеть из готовых модулей `nn.Linear` и `nn.ReLU` и копирует в нее наши веса. Две тонкости:
  1. `nn.Linear` хранит матрицу весов в форме $n_{out} \times n_{in}$ (строка на каждый нейрон) и считает $xW^\top + b$. Наша матрица – $n_{in} \times n_{out}$, поэтому копируем транспонированную: `W_.T`.
  2. Веса модулей `nn` отслеживаются механизмом автоматического дифференцирования (о нем – на следующем занятии). Копирование весов – не часть вычислений сети, поэтому оно выполняется внутри `with torch.no_grad():`.

Функция `init_params` дана готовой: она создает случайные веса по нормальному закону, деленные на $\sqrt{n_{in}}$, – так выходы слоев не растут и не затухают при прохождении сети.

In [ ]:
def init_params(layer_sizes, scale=1.0):
    """Случайные веса: N(0, 1) * scale / sqrt(n_in), смещения – нули."""
    g = torch.Generator().manual_seed(RANDOM_STATE)
    return [(torch.randn(n_in, n_out, generator=g) * scale / n_in ** 0.5, torch.zeros(n_out))
            for n_in, n_out in zip(layer_sizes[:-1], layer_sizes[1:])]


def mlp_forward(x, params):
    for i, (W_, b_) in enumerate(params):
        x = ...                   # <-- линейный слой
        if i < len(params) - 1:
            x = ...               # <-- ReLU везде, кроме последнего слоя
    return x

def to_sequential(params):
    layers = []
    for i, (W_, b_) in enumerate(params):
        lin = nn.Linear(W_.shape[0], W_.shape[1])
        with torch.no_grad():
            lin.weight.copy_(...)  # <-- nn.Linear хранит (n_out, n_in): транспонируйте
            lin.bias.copy_(b_)
        layers.append(lin)
        if i < len(params) - 1:
            layers.append(...)     # <-- модуль ReLU
    return nn.Sequential(*layers)


params_xor = [(W, c), (w.reshape(2, 1), b.reshape(1))]
print(show(mlp_forward(X_xor, params_xor).squeeze(1)), '  ожидается [0.0, 1.0, 1.0, 0.0]')
net_xor = to_sequential(params_xor)
print(net_xor)
print(show(net_xor(X_xor).squeeze(1)), '  ожидается [0.0, 1.0, 1.0, 0.0]')

params_rand = init_params([2, 16, 16, 1])
diff = (mlp_forward(Xm, params_rand) - to_sequential(params_rand)(Xm)).abs().max().item()
print(f'{diff:.1e}', '  ожидается < 1e-05')
print(sum(p.numel() for p in to_sequential(init_params([784, 128, 10])).parameters()),
      '  ожидается 101770 (как count_params)')

Небольшое расхождение с `nn.Linear` (порядка $10^{-7}$) нормально: числа хранятся в `float32`, а PyTorch внутри может складывать произведения в другом порядке, из-за чего округления получаются чуть иными.

**Вопрос С5.** а) Почему `nn.Linear(2, 16).weight` имеет форму $16 \times 2$, а не $2 \times 16$? б) Что вернет `mlp_forward`, если в `params` одна пара? С какой моделью 9 класса это совпадает?

*Ваш ответ:*

**Задание С6. Граница решений на «двух лунах».**

Данные «две луны» – 200 точек двух классов, расположенных двумя изогнутыми полумесяцами. Прямой их не разделить, как и XOR.

- `predict(x, params)` – номер класса. У сети один выход $z$ (логит), вероятность класса 1 равна $\sigma(z)$. Условие $\sigma(z) > 0{,}5$ равносильно $z > 0$, поэтому сигмоиду можно не считать: `(mlp_forward(x, params).squeeze(1) > 0)`. Метод `.squeeze(1)` убирает лишнюю ось: форма $N \times 1$ превращается в $N$. Логическое значение переводим в целое `.long()`.
- `accuracy(y_pred, y_true)` – доля совпадений: сравнение `==` дает тензор из `True`/`False`, `.float()` превращает их в 1 и 0, среднее – это доля.
- `from_sequential(net)` – обратная операция к `to_sequential`: пройти по модулям `net`, для каждого `nn.Linear` взять пару `(m.weight.detach().T, m.bias.detach())`. Метод `.detach()` отцепляет тензор от истории вычислений, `.T` возвращает нашу форму $n_{in} \times n_{out}$.

*Ход эксперимента.* Сначала сеть со случайными весами – ее граница не связана с данными. Затем готовая ячейка обучает ту же архитектуру 500 шагами оптимизатора Adam. Функция потерь `BCEWithLogitsLoss` – это сигмоида и бинарная кросс-энтропия (log-loss) в одной функции, вычисленные устойчиво. Как устроен сам цикл обучения, разберем на занятии Б1.2 – сейчас важно другое: обученные веса мы достаем из `net` и рисуем границу **своей** функцией `mlp_forward`. Если граница совпадает с ответами `net`, наш прямой проход работает так же, как PyTorch.

*Чего ожидать.* Граница обученной сети – ломаная из отрезков (п. 7 теории): в каждой области, где одни и те же нейроны ReLU «включены», сеть линейна.

In [ ]:
def predict(x, params):
    return ...  # <-- (mlp_forward(x, params).squeeze(1) > 0) и перевод в long

def accuracy(y_pred, y_true):
    return ...  # <-- доля совпадений, .float().mean().item()

def from_sequential(net):
    return [...]  # <-- (m.weight.detach().T, m.bias.detach()) для каждого m в net, если это nn.Linear


print(show(mlp_forward(X_xor, from_sequential(net_xor)).squeeze(1)), '  ожидается [0.0, 1.0, 1.0, 0.0]')
print(accuracy(predict(X_xor, params_xor), y_xor.long()), '  ожидается 1.0 (y_hat > 0 совпадает с XOR)')
acc_rand = accuracy(predict(Xm, params_rand), ym)
print(round(acc_rand, 3), '  случайные веса: ожидается заметно ниже 1')

In [ ]:
# Готовая ячейка обучения (подробно разбирается на занятии Б1.2)
torch.manual_seed(RANDOM_STATE)
net = nn.Sequential(nn.Linear(2, 16), nn.ReLU(),
                    nn.Linear(16, 16), nn.ReLU(),
                    nn.Linear(16, 1))
optimizer = torch.optim.Adam(net.parameters(), lr=0.01)
loss_fn = nn.BCEWithLogitsLoss()          # сигмоида + бинарная кросс-энтропия в одной функции
for epoch in range(500):
    optimizer.zero_grad()
    loss = loss_fn(net(Xm).squeeze(1), ym.float())
    loss.backward()
    optimizer.step()
print('потеря после обучения:', round(loss.item(), 3))

params_trained = from_sequential(net)
acc_trained = accuracy(predict(Xm, params_trained), ym)
print(round(acc_trained, 3), '  ожидается не ниже 0.95')
print(torch.equal(predict(Xm, params_trained), (net(Xm).squeeze(1) > 0).long()), '  ожидается True')

In [ ]:
# Готовая ячейка: граница решений своей функцией mlp_forward
def plot_boundary(ax, params, title):
    xx, yy = torch.meshgrid(torch.linspace(-1.6, 2.6, 300), torch.linspace(-1.2, 1.7, 300), indexing='xy')
    grid = torch.stack([xx.ravel(), yy.ravel()], dim=1)
    with torch.no_grad():
        Z = predict(grid, params).reshape(xx.shape)
    ax.contourf(xx, yy, Z, levels=[-0.5, 0.5, 1.5], colors=['#E2E2E6', '#EAE4D2'])
    ax.contour(xx, yy, Z, levels=[0.5], colors=['#8F8260'])
    ax.scatter(Xm[:, 0], Xm[:, 1], c=['#B8AB84' if t else '#1C1C22' for t in ym], s=14)
    ax.set_title(title)

fig, axs = plt.subplots(1, 2, figsize=(9, 3.8))
plot_boundary(axs[0], params_rand, f'Случайные веса, accuracy = {acc_rand:.2f}')
plot_boundary(axs[1], params_trained, f'После обучения, accuracy = {acc_trained:.2f}')
fig.tight_layout(); fig.savefig('moons_boundary.png', dpi=120); plt.close(fig)
display(Image('moons_boundary.png'))

**Вопрос С6.** а) Из каких кусков состоит граница обученной сети с ReLU? Почему она не может быть гладкой кривой? б) Сравните с границами моделей 9 класса на тех же данных: логистическая регрессия, дерево решений, kNN. в) Почему точность на обучающей выборке ничего не говорит о качестве на новых данных?

*Ваш ответ:*

### Задание повышенной сложности (к части 1С)

1. **Устойчивый `log_softmax`.** Напишите `log_softmax(z)` через $z_k - m - \ln \sum_j e^{z_j - m}$, где $m = \max_j z_j$. Сравните с `torch.log(softmax(z))` на $z = (0, -1000)$: у наивной версии получится $-\infty$, у устойчивой – $-1000$. Сверьте с `torch.log_softmax`.
2. **Сеть без активации на «лунах».** Уберите `nn.ReLU` из готовой ячейки обучения и обучите заново. Докажите на бумаге, что граница решений – прямая, и сравните точность с логистической регрессией `sklearn`.
3. **Один скрытый нейрон.** Докажите, что сеть $\hat y = w \cdot \mathrm{ReLU}(a_1x_1 + a_2x_2 + c) + d$ не может решить XOR ни при каких весах. Подсказка: $\hat y$ монотонно зависит от одного числа $s = a_1x_1 + a_2x_2 + c$.
4. **Глубже или шире.** Подберите две сети с примерно одинаковым числом параметров (`count_params`): одну широкую с одним скрытым слоем и одну с тремя узкими слоями. Обучите обе на `make_moons(n_samples=200, noise=0.2)` с `make_moons(..., random_state=0)` в роли теста и сравните точность и вид границ.

In [ ]:
# Место для задания повышенной сложности